# 🎤 Karaoke Mobile — Google Colab MFA Alignment Server

Этот блокнот запускает бесплатный удалённый сервер **Montreal Forced Aligner (MFA)** в Google Colab.
Мобильное приложение подключается к этому серверу через защищённый туннель Cloudflare и отправляет аудио с текстом для точного выравнивания слогов.

### Инструкция:
1. Нажмите **Среда выполнения (Runtime) -> Выполнить всё (Run all)** (или `Ctrl + F9`).
2. Дождитесь появления ссылки вида `https://xxxx.trycloudflare.com` в последней ячейке.
3. Скопируйте ссылку (или отсканируйте QR-код) и вставьте в приложении Karaoke Mobile на телефоне во вкладке **MFA Alignment**!

In [ ]:
#@title 1. Проверка окружения и установка Cloudflared
!curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!pip install -q fastapi uvicorn python-multipart qrcode
print("Cloudflared и базовые пакеты установлены!")

In [ ]:
#@title 2. Установка Conda и Montreal Forced Aligner (MFA)
# Установка micromamba для быстрого развёртывания MFA
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!./bin/micromamba shell init -s bash -p ~/micromamba
!./bin/micromamba create -n mfa -c conda-forge montreal-forced-aligner ffmpeg -y

# Скачивание моделей для русского и английского языков
!~/micromamba/envs/mfa/bin/mfa model download dictionary russian_mfa
!~/micromamba/envs/mfa/bin/mfa model download acoustic russian_mfa
!~/micromamba/envs/mfa/bin/mfa model download dictionary english_mfa
!~/micromamba/envs/mfa/bin/mfa model download acoustic english_mfa
print("MFA и акустические модели готовы к работе!")

In [ ]:
#@title 3. Запись серверного скрипта FastAPI
server_code = """
import subprocess, shutil, tempfile, logging, os
from pathlib import Path
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title='Colab MFA Server')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

@app.get('/')
def status():
    return {'status': 'online', 'mfa': 'active', 'models': ['russian_mfa', 'english_mfa']}

@app.post('/align')
async def align(audio: UploadFile = File(...), transcript: str = Form(...), language: str = Form('russian')):
    with tempfile.TemporaryDirectory(prefix='mfa_run_') as td:
        work = Path(td)
        corpus = work / 'corpus'
        out = work / 'out'
        corpus.mkdir(); out.mkdir()
        
        raw_audio = work / audio.filename
        with open(raw_audio, 'wb') as f:
            shutil.copyfileobj(audio.file, f)
            
        wav = corpus / 'track.wav'
        subprocess.run(['ffmpeg', '-y', '-i', str(raw_audio), '-ar', '16000', '-ac', '1', str(wav)], check=True)
        (corpus / 'track.lab').write_text(' '.join(transcript.strip().split()), encoding='utf-8')
        
        dict_name = 'russian_mfa' if 'ru' in language.lower() else 'english_mfa'
        mfa_bin = os.path.expanduser('~/micromamba/envs/mfa/bin/mfa')
        cmd = [mfa_bin, 'align', str(corpus), dict_name, dict_name, str(out), '--clean', '--single_speaker']
        res = subprocess.run(cmd, capture_output=True, text=True)
        
        tg = out / 'track.TextGrid'
        if not tg.exists():
            return {'success': False, 'message': res.stderr[:800]}
            
        # Parse basic intervals
        lines = tg.read_text(encoding='utf-8', errors='ignore').splitlines()
        words = []
        in_words = False
        i = 0
        while i < len(lines):
            l = lines[i].strip()
            if 'name = "words"' in l: in_words = True
            elif 'name = "phones"' in l: in_words = False
            elif in_words and l.startswith('intervals ['):
                xmin, xmax, text = 0.0, 0.0, ''
                i += 1
                while i < len(lines) and not lines[i].strip().startswith('intervals ['):
                    sub = lines[i].strip()
                    if sub.startswith('xmin ='): xmin = float(sub.split('=')[1])
                    elif sub.startswith('xmax ='): xmax = float(sub.split('=')[1])
                    elif sub.startswith('text ='): text = sub.split('=')[1].strip('" ')
                    i += 1
                if text and text != '<eps>':
                    words.append({'word': text, 'start': xmin, 'end': xmax})
                continue
            i += 1
            
        # Format ASS
        ass_lines = [
            '[Script Info]', 'ScriptType: v4.00+', 'PlayResX: 1920', 'PlayResY: 1080', '',
            '[V4+ Styles]',
            'Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding',
            'Style: Default,Arial,52,&H00FFFFFF&,&H000000FF&,&H00000000&,&H00000000,1,0,0,0,100,100,0,0,1,3,0,5,20,20,20,1', '',
            '[Events]', 'Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text'
        ]
        k_parts = [f'\\k{int(round((w["end"]-w["start"])*100))}{w["word"]}' for w in words]
        if words:
            t_start = f'0:00:{words[0]["start"]:05.2f}'
            t_end = f'0:00:{words[-1]["end"]:05.2f}'
            ass_lines.append(f'Dialogue: 0,{t_start},{t_end},Default,,0,0,0,,{{\\k0}}{" {".join(k_parts)}')
            
        return {'success': True, 'words': words, 'ass_content': '\n'.join(ass_lines)}
"""
with open("mfa_server.py", "w") as f:
    f.write(server_code)
print("Серверный скрипт скомпилирован!")

In [ ]:
#@title 4. Запуск сервера и получение публичной ссылки Cloudflare
import subprocess, time, re, qrcode
from IPython.display import display, HTML

# Запуск uvicorn
!pkill -f uvicorn
!pkill -f cloudflared
subprocess.Popen(["uvicorn", "mfa_server:app", "--port", "8000"])
time.sleep(2)

# Запуск туннеля
p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"], stderr=subprocess.PIPE, text=True)

tunnel_url = None
for _ in range(30):
    line = p.stderr.readline()
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        tunnel_url = m.group(0)
        break
    time.sleep(0.5)

if tunnel_url:
    qr = qrcode.QRCode(box_size=4, border=2)
    qr.add_data(tunnel_url)
    qr.make(fit=True)
    img = qr.make_image(fill_color="black", back_color="white")
    img.save("tunnel_qr.png")
    
    display(HTML(f"""
    <div style='background: #1e1e2e; color: #cdd6f4; padding: 20px; border-radius: 12px; font-family: sans-serif;'>
      <h2 style='color: #a6e3a1; margin-top: 0;'>🟢 Сервер MFA успешно запущен!</h2>
      <p>Вставьте эту ссылку в мобильном приложении Karaoke Mobile:</p>
      <div style='background: #11111b; padding: 12px; border-radius: 8px; font-size: 18px; font-weight: bold; color: #89b4fa; word-break: break-all;'>
        {tunnel_url}
      </div>
    </div>
    """))
    display(img)
else:
    print("Не удалось инициализировать туннель Cloudflare. Попробуйте перезапустить ячейку.")